# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to access, process, and explore the FAIR² dataset of 77 cancer survivors with second primary colorectal cancer using the `mlcroissant` library.

### Dataset Source

The dataset is defined by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` and `pandas` are installed
!pip install -U mlcroissant pandas

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

# Optional: Show other key top-level metadata for exploration
print("\nKey metadata fields:")
for k in ["identifier", "datePublished", "license", "version", "keywords"]:
    if hasattr(metadata, k):
        print(f" - {k}: {getattr(metadata, k)}")

## 2. Data Overview

List all available record sets and their fields with their `@id` values. All references will use Croissant `@id` identifiers for reproducibility.

In [ ]:
# Inspect the available record sets in the dataset

from mlcroissant.structures import RecordSet

record_sets = []

for obj in metadata.record_sets:
    record_sets.append(obj)
    print(f"RecordSet '@id': {obj.id}")
    print(f"  Name: {obj.name}")
    print(f"  Description: {getattr(obj, 'description', 'No description')}")
    
    print("  Fields:")
    for field in obj.fields:
        print(f"    - Field '@id': {field.id} | Name: {field.name}")
    print()

## 3. Data Extraction

Load the contents of specific record sets to DataFrames for further exploration. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare DataFrames for each record set
dataframes = {}

# List of record set '@id's
record_set_ids = [rs.id for rs in record_sets]
print(f"Available record set @ids: {record_set_ids}")

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set '{record_set_id}' with shape {dataframes[record_set_id].shape}")

# Choose the main record set for exploration (first one by default)
main_rs_id = record_set_ids[0]

print(f"\nColumns in main record set ({main_rs_id}):")
print(dataframes[main_rs_id].columns.tolist())

# Show a preview of the data
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's filter, normalize, and group the data using Croissant `@id`s. We'll:
- Select a numeric field (e.g.
    - Age, if present
- Filter for values above a threshold
- Normalize the data
- Group by a categorical field (e.g. sex or anatomical location) if available.

In [ ]:
# Identify a likely numeric field '@id' and a group field '@id'.
# We'll guess based on common variable names if possible.

df = dataframes[main_rs_id]
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if group_field_id is None and (('sex' in col.lower()) or ('gender' in col.lower()) or ('anatomical_location' in col.lower()) or ('site' in col.lower())):
        group_field_id = col

print(f"Numeric field candidate: {numeric_field_id}")
print(f"Group field candidate: {group_field_id}")

if numeric_field_id is not None:
    # Convert to numeric if not already
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.6)  # Use 60th percentile as threshold for demonstration
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by group_field_id, if available
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No candidate numeric field found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields. Here we plot the distribution of the selected numeric field and, if possible, the group-wise averages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Groupwise barplot if group_field available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.barplot(data=df, x=group_field_id, y=numeric_field_id, ci=None)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion

This notebook demonstrated how to access and explore a FAIR² clinical and molecular dataset for cancer survivors with second primary colorectal cancer via its Croissant schema. Using `mlcroissant`, record sets and fields were accessed via their `@id` values, and a range of exploratory analyses showed how to filter, transform, group, and visualize data.

**Key takeaways:**
- Data was fully described and accessed using Croissant `@id`s for maximum reproducibility.
- Exploratory analysis included numeric field filtering, normalization, groupwise comparison, and basic visualization.
- With the mlcroissant + Croissant schema approach, metadata, data, and structure remain tightly integrated in a reusable workflow for scientific analyses.